# Lab 2: Turn retrieved evidence into a grounded HR answer

## Business problem

Retrieval returns policy text, but an employee expects a direct answer with evidence. The application must answer from the retrieved policy, identify its source, and clearly state when the available evidence is insufficient.

## Mission

Connect retrieval to an LLM while keeping each stage visible:

1. retrieve evidence;
2. inspect the evidence;
3. build the grounded prompt;
4. generate the answer;
5. display citations;
6. test a question the policy cannot answer.


## Exercise 1: Build the same index reproducibly

**Mission:** Recreate the Lab 1 pipeline from the same source and configuration.

**Why it matters:** In production this code becomes a versioned indexing job. The notebook keeps it visible so every input can be inspected.


In [ ]:
from pathlib import Path
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

SOURCE_FILE = "hr_policy.txt"
EMBEDDING_MODEL = "text-embedding-3-small"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

policy_text = Path(SOURCE_FILE).read_text(encoding="utf-8")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n## ", "\n\n", "\n", ". ", " ", ""],
)

chunk_texts = splitter.split_text(policy_text)
documents = []

for number in range(len(chunk_texts)):
    documents.append(
        Document(
            page_content=chunk_texts[number],
            metadata={"source": SOURCE_FILE, "chunk": number},
        )
    )

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
vector_store = InMemoryVectorStore.from_documents(documents, embedding=embeddings)

print("Indexed chunks:", len(documents))


## Exercise 2: Retrieve before generating

**Mission:** Make the evidence visible before asking an LLM to answer.

**Experiment:** Change `TOP_K`. A larger value is not automatically better.


In [ ]:
QUESTION = "Can I expense a $300 train ticket without approval?"
TOP_K = 3

retrieved_documents = vector_store.similarity_search_with_score(
    QUESTION,
    k=TOP_K,
)

for document, score in retrieved_documents:
    print("Distance:", round(score, 3))
    print("Source:", document.metadata["source"])
    print("Chunk:", document.metadata["chunk"])
    print(document.page_content)
    print()


### Retrieval checkpoint

Do not generate an answer until you confirm that the reimbursement policy is present. Generation cannot recover evidence that retrieval failed to provide.


## Exercise 3: Build the grounded prompt

**Mission:** Give the model the employee question, the retrieved evidence, and explicit boundaries.


In [ ]:
context_parts = []

for document, score in retrieved_documents:
    citation = "Source: " + document.metadata["source"] + ", chunk " + str(document.metadata["chunk"])
    context_parts.append(citation + "\n" + document.page_content)

context = "\n\n".join(context_parts)

system_message = (
    "You are the company HR policy assistant. "
    "Answer only from the supplied policy context. "
    "If the context does not contain the answer, say: "
    "I cannot find that answer in the available HR policy. "
    "Give a concise answer and list the source citations you used."
)

user_message = "Context:\n" + context + "\n\nEmployee question:\n" + QUESTION

print(user_message)


## Exercise 4: Generate the grounded answer

**Mission:** Ask the generation model to answer only from the retrieved policy.

This lab uses OpenAI for embeddings and Anthropic Claude for generation. Those are separate responsibilities. A company can replace either model after testing compatibility, quality, latency, and cost.


In [ ]:
from anthropic import Anthropic

anthropic_client = Anthropic()
GENERATION_MODEL = "claude-haiku-4-5"

response = anthropic_client.messages.create(
    model=GENERATION_MODEL,
    max_tokens=300,
    system=system_message,
    messages=[{"role": "user", "content": user_message}],
)

answer = response.content[0].text
print(answer)


### Verify the answer

The expected conclusion is that travel under $500 does not require pre-approval. The answer should cite the HR policy rather than presenting the model as the authority.


## Exercise 5: Test the insufficient-evidence path

**Mission:** Confirm that the system does not invent a policy when the source does not contain the answer.


In [ ]:
MISSING_QUESTION = "Does the company reimburse home internet service?"

missing_documents = vector_store.similarity_search(MISSING_QUESTION, k=TOP_K)
missing_context_parts = []

for document in missing_documents:
    citation = "Source: " + document.metadata["source"] + ", chunk " + str(document.metadata["chunk"])
    missing_context_parts.append(citation + "\n" + document.page_content)

missing_context = "\n\n".join(missing_context_parts)
missing_message = "Context:\n" + missing_context + "\n\nEmployee question:\n" + MISSING_QUESTION

missing_response = anthropic_client.messages.create(
    model=GENERATION_MODEL,
    max_tokens=300,
    system=system_message,
    messages=[{"role": "user", "content": missing_message}],
)

print(missing_response.content[0].text)


## Lab 2 checkpoint

You now have the complete RAG request path:

`question -> retrieve -> inspect -> augment -> generate -> cite`

The important operational boundary is visible: retrieval supplies the evidence and the generation model explains that evidence. A fluent answer is not proof that retrieval succeeded.
